In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torchsummary import summary
from torchvision import datasets, transforms

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
data_train = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor()) 
data_test = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())    

batch_size = 32
train_dataloader = DataLoader(data_train, batch_size=batch_size, shuffle=True, drop_last=True)
test_dataloader = DataLoader(data_test, batch_size=batch_size, shuffle=False)

# check out a sample
images, labels = next(iter(train_dataloader))
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")

Image batch shape: torch.Size([32, 1, 28, 28])
Label batch shape: torch.Size([32])


In [9]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        # now in sequential mode
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=10, kernel_size=5, stride=1, padding=1), # output size = np.floor((28 - 5 + 2*1) / 1) + 1 = 26
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # output size = np.floor((26 - 2) / 2) + 1 = 13
            nn.Conv2d(in_channels=10, out_channels=20, kernel_size=5, stride=1, padding=1), # output size = np.floor((13 - 5 + 2*1) / 1) + 1 = 10 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2) # output size = np.floor((10 - 2) / 2) + 1 = 5
        )
        
        self.fnn = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=20*5*5, out_features=50),
            nn.ReLU(),
            nn.Linear(in_features=50, out_features=10),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.fnn(x)
        return x

model = MNIST_CNN().to(device)
summary(model, input_size=(1, 28, 28), device=device.type)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 10, 26, 26]             260
              ReLU-2           [-1, 10, 26, 26]               0
         MaxPool2d-3           [-1, 10, 13, 13]               0
            Conv2d-4           [-1, 20, 11, 11]           5,020
              ReLU-5           [-1, 20, 11, 11]               0
         MaxPool2d-6             [-1, 20, 5, 5]               0
           Flatten-7                  [-1, 500]               0
            Linear-8                   [-1, 50]          25,050
              ReLU-9                   [-1, 50]               0
           Linear-10                   [-1, 10]             510
Total params: 30,840
Trainable params: 30,840
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.16
Params size (MB): 0.12
Estimated Tot

In [ ]:
def train_one_epoch(model: torch.nn.Module,
                   train_dataloader: torch.utils.data.DataLoader,
                   loss_fn: torch.nn.Module,
                   optimizer: torch.optim.Optimizer,
                   device: torch.device): # Added device
    model.train()
    train_loss = 0.0
    train_accuracy = 0.0
    
    for batch, (X, y) in enumerate(train_dataloader):
        # Move data to the specified device
        X, y = X.to(device), y.to(device)
        
        # Forward pass
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        _, predicted = torch.max(y_pred.data, 1)
        train_accuracy += (predicted == y).sum().item() / y.size(0) 

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return train_loss, train_accuracy

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          loss_fn: torch.nn.Module,
          optimizer: torch.optim.Optimizer,
          device: torch.device, # Added device
          epochs: int):
    
    # Move the entire model to the device once at the start
    model.to(device)
    
    losses_epochs = [] 
    train_accuracy_epochs = []
    test_accuracy_epochs = []

    for epoch in range(epochs):
        # Pass device to the epoch function
        train_loss, train_acc = train_one_epoch(model, train_dataloader, loss_fn, optimizer, device)

        avg_train_loss = train_loss / len(train_dataloader)
        avg_train_acc = train_acc / len(train_dataloader)
        losses_epochs.append(avg_train_loss)
        train_accuracy_epochs.append(avg_train_acc)

        # Evaluate on the test set
        model.eval()
        test_accuracy = 0.0
        with torch.no_grad():
            for X, y in test_dataloader:
                # Move test data to device
                X, y = X.to(device), y.to(device)
                y_pred = model(X)

                _, predicted = torch.max(y_pred.data, 1)
                test_accuracy += (predicted == y).sum().item() / y.size(0)

        avg_test_acc = test_accuracy / len(test_dataloader)
        test_accuracy_epochs.append(avg_test_acc)

        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, "
              f"Train Accuracy: {avg_train_acc:.4f}, Test Accuracy: {avg_test_acc:.4f}")

    return losses_epochs, train_accuracy_epochs, test_accuracy_epochs

In [ ]:
model = MNIST_CNN(printtoggle=False)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10
losses, train_accuracy, test_accuracy = train(model, train_dataloader, test_dataloader, loss_fn, optimizer, epochs)